In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import * 

# 1D Examples in periodic conditions

## Complex Ginzburg-Landau equation

In [ ]:
# 1. Define the symbols and the equation
t, x = symbols('t x', real=True)
u = Function('u', complex=True)
# Equation parameters
alpha = 1.0
beta = 0.5  # Imaginary part of diffusion
gamma = 1.0
delta = 0.2  # Imaginary part of the linear term
epsilon = 0.1  # Imaginary part of the nonlinearity
# Complex Ginzburg-Landau equation
linear_diffusion = (alpha + I*beta) * diff(u(t, x), x, x)
linear_growth = (gamma + I*delta) * u(t, x)
nonlinear_term = (1 + I*epsilon) * Abs(u(t, x))**2 * u(t, x)
# Complete equation
equation = Eq(diff(u(t, x), t), linear_diffusion + linear_growth - nonlinear_term)

# 2. Initial condition: complex plane wave
def initial_condition(x_vals):
    A = 1.0  # Amplitude
    k = 0.5  # Wave number
    return A * np.exp(1j * k * x_vals)  # Exact solution in the absence of nonlinearity

# 3. Solver setup
solver = PDESolver(equation, time_scheme='RK4', dealiasing_ratio=2/3)
# Simulation parameters
Lx = 50.0
Nx = 1024
Lt = 10.0
Nt = 1000
# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition
)

# 4. Solve
solver.solve()

# 5. Visualization
ani = solver.animate(component='abs')
HTML(ani.to_jshtml())


## Sine-Gordon equation

In [ ]:
# 1. Define the symbols and the equation
t, x = symbols('t x', real=True)
u = Function('u', real=True)
# Sine-Gordon equation
equation = Eq(diff(u(t, x), t, t), diff(u(t, x), x, x) + sin(u(t, x)))

# 2. Initial condition: stationary kink
def initial_condition(x_vals):
    x0 = 0  # Position of the kink
    return 4 * np.arctan(np.exp(x_vals - x0))  # Kink soliton

# 3. Initial velocity (for a second-order equation in time)
def initial_velocity(x_vals):
    return np.zeros_like(x_vals)  # Initial velocity is zero

# 4. Solver configuration
solver = PDESolver(equation, time_scheme='RK4')
# Simulation parameters
Lx = 20.0
Nx = 1024
Lt = 40.0
Nt = 6000

# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity  # Necessary for second-order equations
)

# 5. Solve
solver.solve()

# 6. Visualization
ani = solver.animate(component='abs')  # Visualize the amplitude of u
HTML(ani.to_jshtml())


## Kuramoto-Sivashinsky equation

In [ ]:
# 1. Symbols
t, x, kx = symbols('t x kx')
u_func = Function('u')
u = u_func(t, x)

# 2. Kuramoto-Sivashinsky equation
# eq = Eq(diff(u, t), -u * diff(u, x) - diff(u, x, 2) - diff(u, x, 4))
eq = Eq(diff(u, t), -u * diff(u, x) - diff(u, x, 2) - 0.1 * diff(u, x, 4))

# 3. Creating the solver
solver = PDESolver(eq, time_scheme='RK4', dealiasing_ratio=2/3)

# 4. Domain parameters
Lx = 2 * np.pi
Nx = 256
Lt = 50.0
Nt = 1000

# 5. Initial condition: smooth random perturbation
rng = np.random.default_rng(42)
# def initial_condition(x_vals):
#    return 0.1 * np.sin(x_vals) + 0.01 * rng.standard_normal(x_vals.shape)

def initial_condition(x_vals):
    return 0.1 * np.cos(3 * x_vals) + 0.01 * rng.normal(size=x_vals.shape)

# 6. Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='periodic',
    initial_condition=initial_condition
)

# 7. Solving
solver.solve()

# 8. Visualization
ani = solver.animate(component='real')
HTML(ani.to_jshtml())


# 1D Examples in Dirichlet conditions

## Non-linear Schrödinger equation

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Linear operator: -∂²/∂x² → ψOp(xi²)
linear_symbol = xi**2

# Non-linear term: |u|² * u
nonlinear_term = Abs(u(t, x))**2 * u(t, x)

# Non-linear Schrödinger equation
equation = Eq(I * diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) + nonlinear_term)

# Create the solver
solver = PDESolver(equation)

# Parameters
Lx = 20.0
Nx = 512
Lt = 10.0
Nt = 1000

# Initial condition: fundamental soliton
def initial_condition(x):
    return 2 / np.cosh(x)  # Order 1 soliton

# Configuration
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

# Solve
solver.solve()

# Visualization
ani = solver.animate(component='abs')
HTML(ani.to_jshtml())


## Heat equation with source

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Linear operator: ∂²/∂x² → ψOp(-xi²)
linear_symbol = -xi**2

# Heat source: Gaussian pulse centered at x=0
source_expr = exp(-(x**2) - t**2)

# Heat equation with source
equation = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) + source_expr)

# Solver
solver = PDESolver(equation)

# Spatial and temporal domain
Lx = 10.0
Nx = 256
Lt = 5.0
Nt = 500

# Initialization: zero temperature initially
def initial_condition(x):
    return np.zeros_like(x)

# Dirichlet boundary conditions: fixed temperature at the ends
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

# Solving
solver.solve()

# Visualization
ani = solver.animate()
HTML(ani.to_jshtml())


## KdV equation

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Linear operator: ∂³/∂x³ → ψOp(i ξ³)
linear_symbol = I * xi**3

# Nonlinear term: u ∂u/∂x
nonlinear_term = u(t, x) * diff(u(t, x), x)

# KdV equation
equation = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) - nonlinear_term)

# Solver
solver = PDESolver(equation)

# Parameters
Lx = 40.0
Nx = 512
Lt = 30.0
Nt = 1500

# Initial condition: KdV soliton
def initial_condition(x):
    return 3 * (1 / np.cosh(x / 2))**2  # Height 3, width 2

# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

# Solve
solver.solve()

# Visualization
ani = solver.animate(component='real')
HTML(ani.to_jshtml())

## Airy equation (not sure that it works - low-amplitude linear waves in shallow water...)

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')
# Linear operator: ∂³/∂x³ → ψOp(i ξ³)
linear_symbol = I * xi**3
# Airy equation
equation = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)))
# Solver
solver = PDESolver(equation)
# Parameters
Lx = 40.0
Nx = 512
Lt = 10.0
Nt = 500
# Initial condition: localized pulse (deformed Gaussian)
def initial_condition(x):
    return np.exp(-x**2) * (1 - 2*x**2)
# Configuration
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)
# Solving
solver.solve()
# Visualization
ani = solver.animate()
HTML(ani.to_jshtml())


## Fisher-KPP equation

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Linear operator: D ∂²/∂x² → ψOp(-D ξ²)
D = 0.5
linear_symbol = -D * xi**2

# Non-linear term: r u(1 - u), where r = growth rate
r = 1.0
nonlinear_term = r * u(t, x) * (1 - u(t, x))

# Fisher-KPP equation
equation = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) + nonlinear_term)

# Creation of the solver
solver = PDESolver(equation)

# Parameters
Lx = 20.0
Nx = 256
Lt = 10.0
Nt = 500

# Initial condition: a wave localized at the center
def initial_condition(x):
    return np.where(np.abs(x) < 1, 1.0, 0.0)

# Configuration
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

# Solving
solver.solve()

# Visualization
ani = solver.animate()
HTML(ani.to_jshtml())


## Klein-Gordon equation with Higgs potential

In [ ]:
# Symbolic variables
t, x, xi = symbols('t x xi', real=True)
u = Function('u')
# Linear operator: ∂²/∂x² → ψOp(-ξ²)
mass = 1.0
nonlinear_coeff = 0.5
linear_symbol = -xi**2 - mass**2
# Non-linear term: λ u³
nonlinear_term = nonlinear_coeff * u(t, x)**3
# Klein-Gordon equation with Higgs potential
equation = Eq(diff(u(t, x), t, 2), psiOp(linear_symbol, u(t, x)) + nonlinear_term)

# Solver
solver = PDESolver(equation)
# Spatial and temporal domain
Lx = 20.0
Nx = 512
Lt = 20.0
Nt = 1000
# Initial condition: localized wave (Gaussian)
def initial_condition(x):
    return np.exp(-(x)**2 / 0.5)
# Initial velocity is zero
def initial_velocity(x):
    return np.zeros_like(x)
# Configuration
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)
# Solving
solver.solve()

# Plot energy
solver.plot_energy()
solver.plot_energy(log=True)

# Visualization
ani = solver.animate()
HTML(ani.to_jshtml())


## Benjamin-Ono equation

In [ ]:
# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Benjamin-Ono Equation: ∂ₜ u = ψOp(-i |ξ|, u) - u ∂ₓ u
symbol = -I * Abs(xi)
nonlinear = u(t, x) * diff(u(t, x), x)
equation = Eq(diff(u(t, x), t), psiOp(symbol, u(t, x)) - nonlinear)

# Initial condition: soliton-like bump
def initial_condition(x):
    return 2 / (1 + x**2)

# Solver configuration
solver = PDESolver(equation, time_scheme='RK4', dealiasing_ratio=2/3)

# Parameters
Lx = 40.0
Nx = 1024
Lt = 30.0
Nt = 2000

# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    initial_condition=initial_condition,
    boundary_condition='nonperiodic'
)

# Solving
solver.solve()

# Visualization
ani = solver.animate(component='real')
HTML(ani.to_jshtml())


## Burgers' equation: Comparative Animation of Viscosity Effects

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sympy import symbols, Function, Eq, diff
from solver import PDESolver
from psiop import psiOp

t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# --- Viscosities (all resolvable with the grid below) ---
nus = [1, 0.1, 0.03, 0.01]
colors = ["r", "g", "b", "k"]

# --- Spatio-temporal domain ---
Lx = 3 * np.pi
Nx = 1024            # dx ≈ 0.009: a Burgers shock is ~2*nu thick, keep it >~ 2*dx
Lt = 10.0
Nt = 5000            # dt = 2e-3 -> advective CFL |u|*dt/dx ≈ 0.2 (nonlinear term is explicit)
n_frames = 100

dt = Lt / Nt
dx = Lx / Nx
print(f"dx = {dx:.4f}, dt = {dt:.1e}, CFL = {dt / dx:.2f}")
if dt / dx > 0.5:
    warnings.warn("Advective CFL > 0.5: the explicit nonlinear step may be unstable.")
for nu in nus:
    if 2 * nu < 2 * dx:   # shock thickness ~ 4*nu/(jump ≈ 2) = 2*nu
        warnings.warn(f"nu={nu}: shock thinner than 2*dx, expect Gibbs oscillations.")

# --- Initial condition: two opposing Gaussians that collide at x = 0 ---
def ic_gaussian(x, x0=0.0, sigma=0.5):
    return np.exp(-0.5 * ((x - x0) / sigma) ** 2)

def initial_condition(x):
    # positive pulse on the left (moves right), negative on the right (moves left)
    return ic_gaussian(x, -1.0) - ic_gaussian(x, +1.0)

# --- Solve for each viscosity ---
fig, ax = plt.subplots()
lines = []
for nu, c in zip(nus, colors):
    # Constant-coefficient psiOp: symbol -nu*xi^2  <=>  nu * u_xx
    linear_symbol = -nu * xi**2
    nonlinear_term = u(t, x) * diff(u(t, x), x)
    eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) - nonlinear_term)

    # psiOp equations are advanced by the first-order ETD1 scheme, so `time_scheme`
    # has no effect here and is left at its default.
    solver = PDESolver(eq)
    solver.setup(
        Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, n_frames=n_frames,
        boundary_condition='nonperiodic',
        boundary_tol=1e-3,          # the solution is ~1e-4 at the edges: clipping is harmless
        initial_condition=initial_condition,
        plot=False,
    )
    sol = np.real(np.array(solver.solve()))      # frames are complex: keep the real part
    line, = ax.plot([], [], c, label=f"nu={nu}")
    lines.append((line, sol))

xgrid = solver.x_grid                            # same grid as the solver (endpoint excluded)
save_interval = max(1, Nt // n_frames)
times = np.arange(len(lines[0][1])) * save_interval * dt   # time of each stored frame

ax.set_title(r"Burgers' equation: $u_t + u u_x = \nu u_{xx}$")
ax.set_xlim(-Lx / 2, Lx / 2)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("x")
ax.legend(loc="upper right")
time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)

def init():
    for line, _ in lines:
        line.set_data([], [])
    time_text.set_text("")
    return [l for l, _ in lines] + [time_text]

def update(frame):
    for line, sol in lines:
        line.set_data(xgrid, sol[frame, :])
    time_text.set_text(f"t = {times[frame]:.2f}")
    return [l for l, _ in lines] + [time_text]

ani = FuncAnimation(fig, update, frames=len(times), init_func=init, blit=True)
plt.close(fig)                                   # avoid a duplicate static figure in Jupyter
display(HTML(ani.to_jshtml()))